# Movie Success Prediction

This notebook predicts whether a movie is a **Flop**, **Average**, or **Hit** from the supplied movie metadata. Run the cells from top to bottom without skipping cells.

## Before you start

Keep this notebook in the project folder, with the spreadsheet at `data/movie_metadata.xlsx`. In Jupyter, select **Kernel → Restart Kernel and Run All Cells** after installing the packages.

In [ ]:
# Run this cell once only if your notebook reports a missing-package error.
%pip install pandas numpy matplotlib seaborn scikit-learn openpyxl

## 1. Import libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

sns.set_theme(style='whitegrid', palette='deep')
RANDOM_STATE = 42
CLASS_ORDER = ['Flop', 'Average', 'Hit']

## 2. Load the dataset

If you placed the Excel file beside the notebook instead of inside `data`, change the path to `Path('movie_metadata.xlsx')`.

In [ ]:
DATA_PATH = Path('data/movie_metadata.xlsx')

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Could not find {DATA_PATH}. Check your notebook folder and file path.')

movies = pd.read_excel(DATA_PATH)
print(f'Rows: {movies.shape[0]:,}')
print(f'Columns: {movies.shape[1]}')
display(movies.head())

## 3. Understand the data

Check variable types, missing values, and duplicate records before building a model.

In [ ]:
movies.info()

missing_values = movies.isna().sum().sort_values(ascending=False)
display(missing_values[missing_values > 0].to_frame('Missing values'))

print(f'Exact duplicate rows: {movies.duplicated().sum()}')
print(f'IMDb score range: {movies.imdb_score.min()} to {movies.imdb_score.max()}')

## 4. Exploratory data analysis

This chart shows the score distribution. The dashed lines are the class boundaries used in the assignment.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=movies, x='imdb_score', bins=30, kde=True, color='#457b9d')
plt.axvline(3, color='#e76f51', linestyle='--', label='Flop/Average boundary')
plt.axvline(6, color='#2a9d8f', linestyle='--', label='Average/Hit boundary')
plt.title('Distribution of IMDb scores')
plt.xlabel('IMDb score')
plt.legend()
plt.show()

## 5. Clean the data and create the target class

The exact boundary rule is: score < 3 = Flop; 3 ≤ score < 6 = Average; score ≥ 6 = Hit. We also create `genre_primary`, the first listed genre.

In [ ]:
model_df = movies.drop_duplicates().copy()
model_df['movie_title'] = model_df['movie_title'].astype(str).str.replace('\u00a0', ' ', regex=False).str.strip()
model_df['genre_primary'] = model_df['genres'].fillna('Unknown').astype(str).str.split('|').str[0]

def classify_score(score):
    if score < 3:
        return 'Flop'
    if score < 6:
        return 'Average'
    return 'Hit'

model_df['success_class'] = model_df['imdb_score'].apply(classify_score)
model_df['success_class'] = pd.Categorical(
    model_df['success_class'], categories=CLASS_ORDER, ordered=True
)

class_counts = model_df['success_class'].value_counts().reindex(CLASS_ORDER)
display(class_counts.to_frame('Movies'))
print(f'Records after duplicate removal: {len(model_df):,}')

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=model_df, x='success_class', order=CLASS_ORDER, hue='success_class', legend=False, palette=['#e76f51', '#e9c46a', '#2a9d8f'])
plt.title('Movie success-class balance')
plt.xlabel('Success class')
plt.ylabel('Number of movies')
plt.show()

## 6. Select features and avoid data leakage

We predict success before release. Therefore, gross earnings, critic/user reviews, vote counts, and movie-page likes are excluded because they are known only after release.

In [ ]:
NUMERIC_FEATURES = [
    'duration', 'director_facebook_likes', 'actor_1_facebook_likes',
    'actor_2_facebook_likes', 'actor_3_facebook_likes',
    'cast_total_facebook_likes', 'facenumber_in_poster',
    'budget', 'title_year', 'aspect_ratio'
]

CATEGORICAL_FEATURES = [
    'color', 'genre_primary', 'language', 'country', 'content_rating'
]

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
X = model_df[FEATURES].copy()
y = model_df['success_class'].copy()
print('Features used:', FEATURES)

## 7. Split the data and build preprocessing

The 80/20 split is stratified, so every class appears in both training and testing sets. Imputation and one-hot encoding are kept inside the pipeline to prevent test-data leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=10))
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, NUMERIC_FEATURES),
    ('categorical', categorical_pipeline, CATEGORICAL_FEATURES)
])

print(f'Training records: {len(X_train):,}')
print(f'Test records: {len(X_test):,}')

## 8. Train and compare three classification models

Accuracy measures overall correctness. Macro F1 is important here because Flop is rare; it gives every class equal weight.

In [ ]:
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, min_samples_leaf=4, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=350, min_samples_leaf=2, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
}

models = {}
comparison_rows = []

for name, classifier in classifiers.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', classifier)
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    models[name] = pipeline
    comparison_rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, predictions),
        'Macro F1': f1_score(y_test, predictions, average='macro')
    })

comparison = pd.DataFrame(comparison_rows).sort_values('Macro F1', ascending=False)
display(comparison.style.format({'Accuracy': '{:.1%}', 'Macro F1': '{:.1%}'}))

## 9. Evaluate the final Random Forest model

Random Forest is the final model required by the project brief. Use the confusion matrix and classification report to explain what it predicts well and where it makes mistakes.

In [ ]:
final_model = models['Random Forest']
rf_predictions = final_model.predict(X_test)

print('Random Forest classification report:')
print(classification_report(y_test, rf_predictions, labels=CLASS_ORDER, zero_division=0))

cm = confusion_matrix(y_test, rf_predictions, labels=CLASS_ORDER)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER)
plt.title('Random Forest confusion matrix')
plt.xlabel('Predicted class')
plt.ylabel('Actual class')
plt.show()

## 10. Feature importance

Feature importance shows which transformed inputs the Random Forest used most often to reduce uncertainty. It shows association, not causation.

In [ ]:
feature_names = final_model.named_steps['preprocessor'].get_feature_names_out()
importance = pd.DataFrame({
    'Feature': [name.replace('numeric__', '').replace('categorical__', '') for name in feature_names],
    'Importance': final_model.named_steps['classifier'].feature_importances_
}).sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(9, 6))
sns.barplot(data=importance.sort_values('Importance'), x='Importance', y='Feature', color='#457b9d')
plt.title('Top 15 Random Forest feature importances')
plt.show()

display(importance)

## 11. Make a sample prediction

Change the values below to describe a proposed movie, then rerun the cell.

In [ ]:
example_movie = pd.DataFrame([{
    'duration': 120,
    'director_facebook_likes': 5000,
    'actor_1_facebook_likes': 20000,
    'actor_2_facebook_likes': 5000,
    'actor_3_facebook_likes': 1000,
    'cast_total_facebook_likes': 30000,
    'facenumber_in_poster': 2,
    'budget': 80000000,
    'title_year': 2026,
    'aspect_ratio': 2.35,
    'color': 'Color',
    'genre_primary': 'Action',
    'language': 'English',
    'country': 'USA',
    'content_rating': 'PG-13'
}])

predicted_class = final_model.predict(example_movie)[0]
probabilities = pd.DataFrame({
    'Class': final_model.named_steps['classifier'].classes_,
    'Probability': final_model.predict_proba(example_movie)[0]
}).sort_values('Probability', ascending=False)

print(f'Predicted movie-success class: {predicted_class}')
display(probabilities.style.format({'Probability': '{:.1%}'}))

## 12. What to say in the presentation

1. I first inspected missing values and duplicate rows.
2. I converted IMDb scores into the required three success classes.
3. I excluded post-release data to avoid data leakage.
4. I used a stratified 80/20 split and compared three classifiers.
5. I selected Random Forest as the final model and evaluated it on unseen test data.
6. I used accuracy, macro F1, a classification report, and a confusion matrix.
7. The rare Flop class is a limitation, so the model should be improved with more low-rated examples.